# Complejidad de comunicación

Este cuaderno ilustra el modelo de comunicación de dos partes, implementa
protocolos para problemas clásicos y demuestra cotas inferiores usando el rango.

**Artículo relacionado:** `04/12-complejidad-de-comunicacion.md`

In [ ]:
import numpy as np
import math
import random

## 1. Matrices de comunicación y rectángulos

La función $f(x,y)$ se representa como una matriz. Cada protocolo de $c$ bits
particiona la matriz en $\leq 2^c$ rectángulos monocromos.

In [ ]:
def build_communication_matrix(f, X, Y):
    """Construye la matriz M donde M[i][j] = f(X[i], Y[j])."""
    return np.array([[f(x, y) for y in Y] for x in X], dtype=int)


def matrix_rank_over_reals(M):
    """Rango de la matriz sobre los reales (usando valores singulares)."""
    return np.linalg.matrix_rank(M.astype(float))


def comm_lower_bound_rank(M):
    """Cota inferior de complejidad de comunicación por rango."""
    r = matrix_rank_over_reals(M)
    return math.ceil(math.log2(r)) if r > 0 else 0


# Problema de igualdad: EQ_n(x,y) = 1 si x == y
n_bits = 4
X = Y = list(range(2**n_bits))

M_eq = build_communication_matrix(lambda x, y: int(x == y), X, Y)
rank_eq = matrix_rank_over_reals(M_eq)
lb_eq = comm_lower_bound_rank(M_eq)

print(f"Problema EQ con entradas de {n_bits} bits ({2**n_bits} valores posibles)")
print(f"Rango de la matriz: {rank_eq}")
print(f"Cota inferior por rango: ceil(log2({rank_eq})) = {lb_eq} bits")
print(f"Protocolo trivial (Alicia envía x): {n_bits} bits")
print(f"Nota: la cota es ajustada (necesita exactamente {n_bits} bits)")
print()

# Matriz de EQ_4
print("Matriz EQ (primeras 8×8 entradas):")
print(M_eq[:8, :8])

## 2. Protocolo aleatorizado para EQ: hashing

In [ ]:
def randomized_eq_protocol(x, y, prime=257, n_rounds=10):
    """
    Protocolo de hashing para EQ:
    - Bob elige primo p aleatorio
    - Bob envía y mod p
    - Alicia verifica si x mod p coincide
    
    Parámetros:
        prime: primo fijo para el hash (en la práctica se elige aleatoriamente)
        n_rounds: número de rondas independientes (amplificación)
    """
    # Generamos primos aleatorios para cada ronda
    small_primes = [11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67]
    
    for _ in range(n_rounds):
        p = random.choice(small_primes)
        # Bob: calcula y mod p, transmite ceil(log2(p)) bits
        bob_hash = y % p
        # Alicia: verifica
        if x % p != bob_hash:
            return False  # Definitivamente distintos
    
    return True  # Probablemente iguales


# Experimento: tasa de falsos positivos (x ≠ y pero se dice iguales)
n_tests = 10000
false_positives = 0
true_positives = 0

random.seed(42)
for _ in range(n_tests):
    x = random.randint(0, 255)
    y = random.randint(0, 255)
    result = randomized_eq_protocol(x, y, n_rounds=3)
    if x != y and result:  # Falso positivo
        false_positives += 1
    if x == y and result:  # Correcto
        true_positives += 1

print("Protocolo aleatorizado para EQ con 3 rondas de hashing:")
print(f"Tests: {n_tests}")
print(f"Falsos positivos (x≠y, resultado=igual): {false_positives} ({false_positives/n_tests*100:.2f}%)")
print(f"Falsos negativos (x=y, resultado=distinto): 0 (protocolo es un-sided)")
print(f"Bits transmitidos por ronda: ~{math.ceil(math.log2(47))} bits (hash mod primo pequeño)")
print(f"Total bits por protocolo (3 rondas): ~{3 * math.ceil(math.log2(47))} bits")
print(f"Protocolo determinista necesita: {n_bits} bits")
print("→ El protocolo aleatorizado usa O(log n) bits vs O(n) deterministas")

## 3. Problema del producto interno: IP_n

In [ ]:
def inner_product_mod2(x_bits, y_bits):
    """IP(x,y) = dot(x,y) mod 2 donde x,y son vectores binarios."""
    return sum(a*b for a, b in zip(x_bits, y_bits)) % 2


def int_to_bits(n, length):
    """Convierte entero n a lista de bits de longitud dada."""
    return [(n >> i) & 1 for i in range(length)]


# Construir matriz de IP para n=3
n_ip = 3
N = 2**n_ip

X_ip = [int_to_bits(i, n_ip) for i in range(N)]
Y_ip = [int_to_bits(j, n_ip) for j in range(N)]

M_ip = np.array([[inner_product_mod2(x, y) for y in Y_ip] for x in X_ip])
rank_ip = matrix_rank_over_reals(M_ip)
lb_ip = comm_lower_bound_rank(M_ip)

print(f"Problema IP_n con n={n_ip} bits")
print(f"Matriz IP_{n_ip} ({N}×{N}):")
print(M_ip)
print(f"Rango sobre R: {rank_ip}")
print(f"Cota inferior por rango: {lb_ip} bits")
print(f"Protocolo trivial: {n_ip} bits (Alicia envía x)")
print()

# Tabla de rangos para distintos n
print(f"{'n':<5} {'2^n':<8} {'rango':<10} {'lb_rango':<12} {'lb_trivial'}")
for k in range(1, 5):
    Nk = 2**k
    Xk = [int_to_bits(i, k) for i in range(Nk)]
    Yk = [int_to_bits(j, k) for j in range(Nk)]
    Mk = np.array([[inner_product_mod2(x, y) for y in Yk] for x in Xk])
    rk = matrix_rank_over_reals(Mk)
    lbk = comm_lower_bound_rank(Mk)
    print(f"{k:<5} {Nk:<8} {rk:<10} {lbk:<12} {k}")

## 4. Protocolo óptimo para Mayor-Que (GT)

In [ ]:
def gt_protocol_deterministic(x, y, n_bits):
    """
    Protocolo para GT_n(x,y) = [x > y].
    
    Estrategia: búsqueda binaria sobre los bits más significativos.
    Alicia y Bob encuentran el primer bit donde difieren.
    
    Comunicación: O(log n) bits si se usa búsqueda binaria,
    o trivialmente n+1 bits enviando x.
    
    Esta implementación muestra el protocolo trivial y el de búsqueda binaria.
    """
    x_bits = [(x >> (n_bits - 1 - i)) & 1 for i in range(n_bits)]  # MSB primero
    y_bits = [(y >> (n_bits - 1 - i)) & 1 for i in range(n_bits)]
    
    bits_sent = 0
    
    # Protocolo: Alicia comparte cada bit de x; Bob responde si coincide con y
    # En el primer bit donde difieren, se determina quién es mayor
    for i in range(n_bits):
        bits_sent += 1  # Alicia envía bit i de x
        if x_bits[i] != y_bits[i]:
            bits_sent += 1  # Bob responde "difieren"
            return x_bits[i] > y_bits[i], bits_sent
        # Si coinciden, continuamos
    
    return False, bits_sent  # x == y, no x > y


# Estadísticas del protocolo GT
n_bits_gt = 8
total_bits = 0
n_samples = 1000
correct = 0

random.seed(42)
for _ in range(n_samples):
    x = random.randint(0, 2**n_bits_gt - 1)
    y = random.randint(0, 2**n_bits_gt - 1)
    result, bits = gt_protocol_deterministic(x, y, n_bits_gt)
    total_bits += bits
    if result == (x > y):
        correct += 1

print(f"Protocolo para GT con entradas de {n_bits_gt} bits")
print(f"Bits enviados (media): {total_bits/n_samples:.2f}")
print(f"Bits enviados (peor caso): {2*n_bits_gt} (cuando x=y)")
print(f"Corrección: {correct}/{n_samples}")
print()
print("Distribución de bits enviados:")

bits_dist = {}
random.seed(42)
for _ in range(n_samples):
    x = random.randint(0, 2**n_bits_gt - 1)
    y = random.randint(0, 2**n_bits_gt - 1)
    _, bits = gt_protocol_deterministic(x, y, n_bits_gt)
    bits_dist[bits] = bits_dist.get(bits, 0) + 1

for b in sorted(bits_dist):
    print(f"  {b} bits: {bits_dist[b]} veces ({bits_dist[b]/n_samples*100:.1f}%)")

## 5. Resumen: cota de rango para distintos problemas

In [ ]:
problems = [
    ("EQ_4",   lambda x, y: int(x == y),  list(range(16)), list(range(16))),
    ("AND_4",  lambda x, y: int(x & y > 0), list(range(16)), list(range(16))),
    ("GT_4",   lambda x, y: int(x > y),   list(range(16)), list(range(16))),
    ("DISJ_4", lambda x, y: int((x & y) == 0), list(range(16)), list(range(16))),
]

print(f"{'Problema':<12} {'Rango':<10} {'LB rango':<12} {'Trivial'}")
print('-' * 45)
for name, f, X, Y in problems:
    M = build_communication_matrix(f, X, Y)
    r = matrix_rank_over_reals(M)
    lb = comm_lower_bound_rank(M)
    trivial = math.ceil(math.log2(len(X)))
    print(f"{name:<12} {r:<10.0f} {lb:<12} {trivial}")

print()
print("Observación: EQ tiene rango máximo (= tamaño de la entrada) → cota de rango ajustada.")
print("AND tiene rango bajo porque la columna de ceros dominan la estructura.")